# Hawkes/SVMHJD Discrete Log-Return Demo

This notebook is a guarded public workflow for the Hawkes/SVMHJD discrete benchmark. Log-return tokenization is used because the jump process is signed and lower-tail behaviour matters directly in return space. The registered discrete setup uses the hidden128 log-return cb64 tokenizer, keeps the additive AR prior as the required jump-profile ablation, and selects the causal conv-transformer k3 prior under the balanced/smooth research profile. The tiny conv-transformer is an optional efficiency candidate that improves jump-count and inter-arrival means but loses the balanced smooth profile. Hawkes/SVMHJD remains `research_candidate` metadata with `public_default: false`.

The notebook prints reproducible commands and optionally reads local artefacts. It does not train or evaluate full models by default and does not require local checkpoints.


## Setup

Load repository helpers and optional plotting utilities. All heavy work is guarded by explicit flags.


In [ ]:
from __future__ import annotations

import json
import os
import shlex
import sys
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT.parent != REPO_ROOT and not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent

NOTEBOOK_DIR = REPO_ROOT / "notebooks" / "discrete"
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-time-causal-vae")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import Markdown, display

try:
    from torchview import draw_graph
except ImportError:
    draw_graph = None

try:
    from scipy.spatial import Voronoi, voronoi_plot_2d
except Exception:
    Voronoi = None
    voronoi_plot_2d = None

try:
    from sklearn.decomposition import PCA
except Exception:
    PCA = None

AUTO_SELECT_MODEL = True
MODEL_REGISTRY_PATH = "../../trained_models/model_registry.yaml"
RUN_SMOKE = True
RUN_FULL = False
RUN_TRAINING = False
RUN_EVALUATION = False
RUN_HEAVY = False

EXPERIMENT_ID = "hawkes_jump"
FAMILY = "discrete"
N_SAMPLE_SMOKE = 128
N_SAMPLE_FULL = 1024
N_SAMPLE = N_SAMPLE_FULL if RUN_FULL else N_SAMPLE_SMOKE

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True})

print(f"repo_root={REPO_ROOT}")
print(f"torchview_available={draw_graph is not None}")
print(f"voronoi_available={Voronoi is not None and voronoi_plot_2d is not None}")
print(f"run_training={RUN_TRAINING}")
print(f"run_evaluation={RUN_EVALUATION}")

In [ ]:
def repo_path(path: str | Path) -> Path:
    path = Path(path)
    return path if path.is_absolute() else REPO_ROOT / path


def resolve_parameter_path(path: str | Path) -> Path:
    raw_path = Path(path)
    if raw_path.is_absolute():
        return raw_path
    notebook_relative = (NOTEBOOK_DIR / raw_path).resolve()
    if notebook_relative.exists():
        return notebook_relative
    return (REPO_ROOT / raw_path).resolve()


def display_path(path: str | Path) -> str:
    path = Path(path)
    try:
        return str(path.resolve().relative_to(REPO_ROOT))
    except ValueError:
        return str(path)


def shell_join(parts: Sequence[str | Path]) -> str:
    return shlex.join([str(part) for part in parts])


def read_yaml(path: str | Path) -> dict[str, Any]:
    with repo_path(path).open("r", encoding="utf-8") as handle:
        payload = yaml.safe_load(handle) or {}
    return payload if isinstance(payload, dict) else {}


def read_json_if_present(path: str | Path) -> dict[str, Any] | None:
    resolved = repo_path(path)
    if not resolved.exists():
        return None
    with resolved.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)
    return payload if isinstance(payload, dict) else None


def load_torch_mapping_if_present(path: str | Path) -> dict[str, Any] | None:
    resolved = repo_path(path)
    if not (RUN_HEAVY and resolved.exists()):
        return None
    payload = torch.load(resolved, map_location="cpu", weights_only=True)
    return dict(payload) if isinstance(payload, Mapping) else None


def metric_value(metrics: Mapping[str, Any], *keys: str) -> float | None:
    for key in keys:
        value = metrics.get(key)
        if isinstance(value, Mapping):
            value = value.get("mean")
        if isinstance(value, int | float):
            return float(value)
    return None


def metric_std(metrics: Mapping[str, Any], *keys: str) -> float | None:
    for key in keys:
        value = metrics.get(key)
        if isinstance(value, Mapping) and isinstance(value.get("std"), int | float):
            return float(value["std"])
        std_value = metrics.get(f"{key}_std")
        if isinstance(std_value, int | float):
            return float(std_value)
    return None


def format_metric(metrics: Mapping[str, Any], *keys: str) -> str:
    value = metric_value(metrics, *keys)
    std = metric_std(metrics, *keys)
    if value is None:
        return ""
    return f"{value:.4g}" if std is None else f"{value:.4g} +/- {std:.2g}"


def pca_project(vectors: np.ndarray) -> np.ndarray:
    centred = vectors - vectors.mean(axis=0, keepdims=True)
    if PCA is not None:
        return PCA(n_components=2).fit_transform(centred)
    _, _, vh = np.linalg.svd(centred, full_matrices=False)
    return centred @ vh[:2].T

## Registry Selection

Select the registered Hawkes/SVMHJD discrete research candidate and inspect the required additive AR ablation, optional tiny efficiency candidate, sampling policy, tokenizer config, and prior config. The notebook uses registry selection by default, so no code change is needed unless a user explicitly chooses another candidate or profile.


In [ ]:
from time_causal_vae.experiments.model_registry import load_registry, select_registered_model

registry_path = resolve_parameter_path(MODEL_REGISTRY_PATH)
registry = load_registry(registry_path) if AUTO_SELECT_MODEL else {}
selected = select_registered_model(registry, experiment=EXPERIMENT_ID, family=FAMILY)
hawkes_registry = registry.get("experiments", {}).get(EXPERIMENT_ID, {})
discrete_section = (
    hawkes_registry.get("discrete", {}) if isinstance(hawkes_registry, Mapping) else {}
)
discrete_candidates = (
    discrete_section.get("candidates", {}) if isinstance(discrete_section, Mapping) else {}
)
required_ablation_id = (
    discrete_section.get("required_ablation") if isinstance(discrete_section, Mapping) else None
)
required_ablation = (
    discrete_candidates.get(required_ablation_id, {})
    if isinstance(discrete_candidates, Mapping)
    else {}
)

TOKENIZER_CONFIG = Path(
    selected.tokenizer_config
    or "configs/experiments/hawkes_jump_causal_vq_tokenizer_hidden128_logreturn_cb64.yaml"
)
SELECTED_PRIOR_CONFIG = Path(
    selected.prior_config
    or "configs/experiments/hawkes_jump_causal_token_prior_hidden128_logreturn_cb64_conv_transformer.yaml"
)
ADDITIVE_PRIOR_CONFIG = Path(
    required_ablation.get(
        "prior_config",
        "configs/experiments/hawkes_jump_causal_token_prior_hidden128_logreturn_cb64_additive.yaml",
    )
)
TOKENIZER_DIR = Path(
    "outputs/hawkes_jump_logreturn_robustness/tokenizers/hawkes_jump_causal_vq_tokenizer_hidden128_logreturn_cb64_seed0"
)
TOKEN_DATA_DIR = Path(
    "outputs/hawkes_jump_logreturn_robustness/tokens/hawkes_jump_causal_vq_tokenizer_hidden128_logreturn_cb64_seed0"
)
ADDITIVE_PRIOR_DIR = Path("outputs/hawkes_jump_logreturn_robustness/priors/additive_seed0")
CONV_PRIOR_DIR = Path("outputs/hawkes_jump_logreturn_robustness/priors/conv_transformer_seed0")
ADDITIVE_EVAL_DIR = Path("outputs/hawkes_jump_logreturn_robustness/evaluations/additive_seed0")
CONV_EVAL_DIR = Path("outputs/hawkes_jump_logreturn_robustness/evaluations/conv_transformer_seed0")

rows = [
    {
        "role": "selected research candidate",
        "candidate": selected.candidate_id,
        "tokenizer_config": selected.tokenizer_config,
        "prior_config": selected.prior_config,
        "sampling": selected.sampling,
        "status": selected.candidate.get("status"),
        "public_default": selected.candidate.get("public_default"),
    },
    {
        "role": "required ablation",
        "candidate": required_ablation_id,
        "tokenizer_config": required_ablation.get("tokenizer_config"),
        "prior_config": required_ablation.get("prior_config"),
        "sampling": required_ablation.get("sampling"),
        "status": required_ablation.get("status"),
        "public_default": required_ablation.get("public_default"),
    },
]
display(pd.DataFrame(rows))

## Config Inspection

The tokenizer is hidden128 with cb64 log-return codes. The additive AR prior remains the required jump-profile ablation, the tiny conv-transformer is an optional efficiency candidate, and the causal conv-transformer k3 prior is selected under the balanced/smooth research profile.


In [ ]:
config_rows: list[dict[str, Any]] = []
for label, path in [
    ("tokenizer", TOKENIZER_CONFIG),
    ("additive_prior", ADDITIVE_PRIOR_CONFIG),
    ("conv_transformer_prior", SELECTED_PRIOR_CONFIG),
]:
    payload = read_yaml(path)
    experiment = (
        payload.get("experiment", {}) if isinstance(payload.get("experiment"), Mapping) else {}
    )
    model = payload.get("model", {}) if isinstance(payload.get("model"), Mapping) else {}
    training = payload.get("training", {}) if isinstance(payload.get("training"), Mapping) else {}
    config_rows.append(
        {
            "role": label,
            "path": display_path(path),
            "exists": repo_path(path).exists(),
            "experiment": experiment.get("name"),
            "seed": experiment.get("seed"),
            "family": model.get("family"),
            "prior_type": model.get("prior_type"),
            "codebook_size": model.get("codebook_size"),
            "epochs": training.get("epochs"),
        }
    )

display(pd.DataFrame(config_rows))

## Tokenizer Commands

The commands below train a smoke tokenizer, extract token indices, and evaluate the tokenizer. They are printed only; generated tokens and summaries belong under ignored `outputs/` paths.


In [ ]:
tokenizer_train_smoke_command = [
    "poetry",
    "run",
    "tcvae-train-tokenizer",
    "--config",
    display_path(TOKENIZER_CONFIG),
    "--output-dir",
    str(TOKENIZER_DIR.parent),
    "--epochs",
    "1",
    "--no-wandb",
    "--dry-run",
]
token_extract_command = [
    "poetry",
    "run",
    "python",
    "scripts/extract_token_indices.py",
    "--config",
    display_path(TOKENIZER_CONFIG),
    "--tokenizer-dir",
    str(TOKENIZER_DIR),
    "--output-dir",
    str(TOKEN_DATA_DIR),
    "--base-data-dir",
    "data/processed",
]
tokenizer_eval_command = [
    "poetry",
    "run",
    "tcvae-evaluate-tokenizer",
    "--config",
    display_path(TOKENIZER_CONFIG),
    "--tokenizer-dir",
    str(TOKENIZER_DIR),
    "--output-dir",
    str(TOKENIZER_DIR / "evaluation"),
    "--base-data-dir",
    "data/processed",
]

display(
    pd.DataFrame(
        [
            {
                "purpose": "train tokenizer smoke",
                "command": shell_join(tokenizer_train_smoke_command),
            },
            {"purpose": "extract token indices", "command": shell_join(token_extract_command)},
            {"purpose": "evaluate tokenizer", "command": shell_join(tokenizer_eval_command)},
        ]
    )
)

## Prior Commands

The additive AR ablation, conv-transformer k3 candidate, and optional tiny efficiency candidate share the same tokenizer and token-data convention. Evaluation commands use the Hawkes-specific evaluator when it is available, with the generic token-prior evaluator shown as a fallback on this public branch. The notebook does not hard-code tiny as the default.


In [ ]:
additive_train_command = [
    "poetry",
    "run",
    "tcvae-train-token-prior",
    "--config",
    display_path(ADDITIVE_PRIOR_CONFIG),
    "--output-dir",
    str(ADDITIVE_PRIOR_DIR),
    "--epochs",
    "1",
    "--no-wandb",
    "--dry-run",
]
conv_train_command = [
    "poetry",
    "run",
    "tcvae-train-token-prior",
    "--config",
    display_path(SELECTED_PRIOR_CONFIG),
    "--output-dir",
    str(CONV_PRIOR_DIR),
    "--epochs",
    "1",
    "--no-wandb",
    "--dry-run",
]
additive_eval_hawkes_command = [
    "poetry",
    "run",
    "python",
    "scripts/evaluate_hawkes_jump_token_prior.py",
    "--config",
    display_path(ADDITIVE_PRIOR_CONFIG),
    "--prior-dir",
    str(ADDITIVE_PRIOR_DIR),
    "--tokenizer-dir",
    str(TOKENIZER_DIR),
    "--output-dir",
    str(ADDITIVE_EVAL_DIR),
    "--n-sample",
    str(N_SAMPLE),
    "--temperature",
    "1.0",
]
conv_eval_hawkes_command = [
    "poetry",
    "run",
    "python",
    "scripts/evaluate_hawkes_jump_token_prior.py",
    "--config",
    display_path(SELECTED_PRIOR_CONFIG),
    "--prior-dir",
    str(CONV_PRIOR_DIR),
    "--tokenizer-dir",
    str(TOKENIZER_DIR),
    "--output-dir",
    str(CONV_EVAL_DIR),
    "--n-sample",
    str(N_SAMPLE),
    "--temperature",
    "1.0",
]
additive_eval_generic_command = [
    "poetry",
    "run",
    "tcvae-evaluate-token-prior",
    "--config",
    display_path(ADDITIVE_PRIOR_CONFIG),
    "--prior-dir",
    str(ADDITIVE_PRIOR_DIR),
    "--tokenizer-dir",
    str(TOKENIZER_DIR),
    "--output-dir",
    str(ADDITIVE_EVAL_DIR),
    "--n-sample",
    str(N_SAMPLE),
    "--temperature",
    "1.0",
]
conv_eval_generic_command = [
    "poetry",
    "run",
    "tcvae-evaluate-token-prior",
    "--config",
    display_path(SELECTED_PRIOR_CONFIG),
    "--prior-dir",
    str(CONV_PRIOR_DIR),
    "--tokenizer-dir",
    str(TOKENIZER_DIR),
    "--output-dir",
    str(CONV_EVAL_DIR),
    "--n-sample",
    str(N_SAMPLE),
    "--temperature",
    "1.0",
]

rows = [
    {"purpose": "train additive AR ablation", "command": shell_join(additive_train_command)},
    {"purpose": "train conv-transformer k3", "command": shell_join(conv_train_command)},
    {
        "purpose": "evaluate additive AR with Hawkes evaluator",
        "command": shell_join(additive_eval_hawkes_command),
    },
    {
        "purpose": "evaluate conv-transformer k3 with Hawkes evaluator",
        "command": shell_join(conv_eval_hawkes_command),
    },
]
if not repo_path("scripts/evaluate_hawkes_jump_token_prior.py").exists():
    rows.extend(
        [
            {
                "purpose": "available additive AR generic fallback",
                "command": shell_join(additive_eval_generic_command),
            },
            {
                "purpose": "available conv-transformer generic fallback",
                "command": shell_join(conv_eval_generic_command),
            },
        ]
    )

display(pd.DataFrame(rows))

if RUN_TRAINING:
    raise RuntimeError(
        "RUN_TRAINING is intentionally guarded. Run the printed commands in a terminal when needed."
    )
if RUN_EVALUATION:
    raise RuntimeError(
        "RUN_EVALUATION is intentionally guarded. Run the printed evaluator commands in a terminal when needed."
    )

## Optional Local-Output Comparison

If local evaluation summaries are present, compare additive AR, conv-transformer k3, and any explicitly selected optional efficiency candidate across smooth metrics, jump metrics, VaR/ES, and token diagnostics. Otherwise, the notebook prints the exact commands to generate the registry-selected k3 and required additive AR runs.


In [ ]:
evaluation_specs = {
    "additive_ar": {
        "summary": ADDITIVE_EVAL_DIR / "evaluation_summary.json",
        "command": additive_eval_hawkes_command,
    },
    "conv_transformer_k3": {
        "summary": CONV_EVAL_DIR / "evaluation_summary.json",
        "command": conv_eval_hawkes_command,
    },
}

summary_rows: list[dict[str, str]] = []
missing_commands: list[str] = []
for label, spec in evaluation_specs.items():
    summary = read_json_if_present(spec["summary"])
    if summary is None:
        missing_commands.append(shell_join(spec["command"]))
        continue
    metrics = summary.get("metrics", summary)
    if not isinstance(metrics, Mapping):
        metrics = {}
    summary_rows.append(
        {
            "candidate": label,
            "MMD": format_metric(metrics, "mmd"),
            "SWD": format_metric(metrics, "swd"),
            "Terminal W1": format_metric(metrics, "terminal_wasserstein"),
            "Volatility W1": format_metric(metrics, "volatility_wasserstein"),
            "Drawdown W1": format_metric(metrics, "drawdown_wasserstein"),
            "Jump-count W1": format_metric(metrics, "jump_count_wasserstein"),
            "Inter-arrival W1": format_metric(metrics, "inter_arrival_wasserstein"),
            "Jump-size W1": format_metric(metrics, "jump_size_wasserstein"),
            "VaR 1%": format_metric(metrics, "var_01", "lower_tail_var_q01"),
            "ES 1%": format_metric(metrics, "es_01", "lower_tail_es_q01"),
            "Active codes": format_metric(
                metrics, "sampled_active_codes", "sampled_token_active_code_count"
            ),
            "Sampled perplexity": format_metric(
                metrics, "sampled_codebook_perplexity", "sampled_token_codebook_perplexity"
            ),
            "Transition L1": format_metric(metrics, "transition_matrix_l1"),
            "Run-length W1": format_metric(metrics, "run_length_wasserstein"),
        }
    )

if summary_rows:
    display(pd.DataFrame(summary_rows))
else:
    display(Markdown("No local evaluation summaries found. Generate them with:"))
    for command in missing_commands:
        display(Markdown(f"```bash\n{command}\n```"))

## Codebook and Geometry Diagnostics

When local tokenizer and token data exist, this section can load codebook vectors, project them with PCA, draw Voronoi cells when SciPy supports the projection, and compare code usage in jump and non-jump windows. The section is skipped on clean checkouts.


In [ ]:
def load_codebook_vectors(tokenizer_dir: Path) -> np.ndarray | None:
    checkpoint_path = repo_path(tokenizer_dir) / "tokenizer.pt"
    if not (RUN_HEAVY and checkpoint_path.exists()):
        return None
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    if not isinstance(checkpoint, Mapping):
        return None
    state = checkpoint.get("model_state_dict")
    if not isinstance(state, Mapping):
        return None
    for key in ("quantizer.backend._codebook.embed", "quantizer.backend._codebook.embed_avg"):
        value = state.get(key)
        if isinstance(value, torch.Tensor):
            vectors = value.detach().cpu().float()
            if vectors.ndim == 3 and vectors.shape[0] == 1:
                vectors = vectors[0]
            return vectors.numpy()
    return None


def load_token_usage(token_data_dir: Path) -> tuple[np.ndarray, np.ndarray] | None:
    token_path = repo_path(token_data_dir) / "eval_tokens.pt"
    if not (RUN_HEAVY and token_path.exists()):
        return None
    payload = torch.load(token_path, map_location="cpu", weights_only=True)
    if not isinstance(payload, Mapping):
        return None
    indices = payload.get("indices")
    data = payload.get("data")
    if not isinstance(indices, torch.Tensor) or not isinstance(data, torch.Tensor):
        return None
    flattened_indices = indices.detach().cpu().long().reshape(-1)
    returns = data.detach().cpu().float()
    if returns.ndim == 3 and returns.shape[-1] == 1:
        returns = returns[..., 0]
    jump_threshold = torch.quantile(returns.abs().reshape(-1), 0.99)
    jump_mask = returns.abs() >= jump_threshold
    if jump_mask.shape != indices.shape:
        jump_mask = jump_mask.reshape(indices.shape)
    codebook_size = max(
        int(flattened_indices.max().item()) + 1 if flattened_indices.numel() else 0, 64
    )
    jump_counts = torch.bincount(indices[jump_mask].reshape(-1).long(), minlength=codebook_size)[
        :codebook_size
    ]
    non_jump_counts = torch.bincount(
        indices[~jump_mask].reshape(-1).long(), minlength=codebook_size
    )[:codebook_size]
    return jump_counts.numpy(), non_jump_counts.numpy()


codebook_vectors = load_codebook_vectors(TOKENIZER_DIR)
usage_pair = load_token_usage(TOKEN_DATA_DIR)
if codebook_vectors is None:
    display(
        Markdown(
            f"No local tokenizer codebook loaded from `{display_path(repo_path(TOKENIZER_DIR))}`. "
            "Set `RUN_HEAVY=True` after training/extracting local artefacts to enable geometry plots."
        )
    )
else:
    projected = pca_project(codebook_vectors)
    fig, ax = plt.subplots(figsize=(6.2, 5.0))
    ax.scatter(projected[:, 0], projected[:, 1], s=42, alpha=0.85)
    for index, (x_value, y_value) in enumerate(projected):
        ax.text(x_value, y_value, str(index), fontsize=7, alpha=0.72)
    if Voronoi is not None and voronoi_plot_2d is not None and projected.shape[0] >= 4:
        try:
            vor = Voronoi(projected)
            voronoi_plot_2d(
                vor, ax=ax, show_vertices=False, line_width=0.6, line_alpha=0.35, point_size=0
            )
        except Exception as exc:
            print(f"Voronoi skipped: {exc}")
    ax.set_title("Hidden128 cb64 codebook PCA")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    display(fig)
    plt.close(fig)

if usage_pair is None:
    display(
        Markdown(f"No local token usage loaded from `{display_path(repo_path(TOKEN_DATA_DIR))}`.")
    )
else:
    jump_counts, non_jump_counts = usage_pair
    total = jump_counts + non_jump_counts
    top_codes = np.argsort(total)[-20:][::-1]
    usage_table = pd.DataFrame(
        {
            "code": top_codes,
            "jump_window_count": jump_counts[top_codes],
            "non_jump_window_count": non_jump_counts[top_codes],
        }
    )
    display(usage_table)
    fig, ax = plt.subplots(figsize=(9, 4.2))
    x_positions = np.arange(len(top_codes))
    ax.bar(x_positions - 0.2, jump_counts[top_codes], width=0.4, label="jump windows")
    ax.bar(x_positions + 0.2, non_jump_counts[top_codes], width=0.4, label="non-jump windows")
    ax.set_xticks(x_positions)
    ax.set_xticklabels([str(code) for code in top_codes], rotation=90)
    ax.set_title("Top code usage by jump-window proxy")
    ax.set_xlabel("Code")
    ax.set_ylabel("Count")
    ax.legend()
    display(fig)
    plt.close(fig)

## Optional Torchview Diagram

The diagram is guarded because it requires optional dependencies and model instantiation. It is intended for local architecture inspection, not for committed outputs.


In [ ]:
if draw_graph is None:
    display(Markdown("`torchview` is not installed; diagram generation is skipped."))
elif not RUN_HEAVY:
    display(Markdown("Torchview diagram skipped because `RUN_HEAVY=False`."))
else:
    display(Markdown("Instantiate the tokenizer or prior locally before calling `draw_graph`."))